# Titanic Survival Prediction V6 — 综合最佳实践

## V6 版本说明

V6 是 V1-V5 所有研究的**综合版本**，将 V4 的最佳架构、[Social Patterns (0.78708)](https://www.kaggle.com/code/konstantinmasich/titanic-social-patterns-eda-survival-modeling) 的 OOF Target Encoding 和关键特征、[CV-v3 (0.80382)](https://www.kaggle.com/code/konstantinmasich/titanic-cv-v3) 的激进 LGBM 参数和 Log-loss 权重优化融合为一体。

### 设计哲学
- **V4 的正确方向保留**：多模型 Ensemble、多样化算法、CV 一致性
- **V5 的错误方向纠正**：放弃单模型、放弃保守参数、放弃 accuracy-based 权重搜索
- **外部最佳实践引入**：OOF Target Encoding（替代 V3 的 LOO 泄漏编码）、WomanOrChild 等已验证特征
- **激进但经过验证的策略**：5000 棵树 LGBM、Log-loss 优化、10 折 CV

### 预期
综合这些经过独立验证的最佳实践，V6 应该超越 V4 的 0.77751（当前最高 LB 分数）。

## V1-V5 问题总结表

| 版本 | 核心问题 | LB 分数 | 教训 |
|------|---------|---------|------|
| **V1** | 默认参数 ensemble + Pclass 未 one-hot（树模型无法正确利用有序类别） | 0.75837 | 永远不要用默认参数；有序类别必须 one-hot 或显式处理 |
| **V2** | 修复了 V1 的 bug，但预测几乎不变（仅 16/418 行变化） | 0.75837 | Bug 修复 ≠ 模型改善；需要从架构层面改进 |
| **V3** | LOO 编码导致 CV 泄漏（CV 0.89 vs LB 0.77 gap=0.12）、Optuna 基于虚假信号调参 | 0.77033 | **LOO 编码在 CV 中泄漏标签**；CV-LB gap > 0.03 就是泄漏信号 |
| **V4** | 57 特征过拟合（891 行数据）、6 棵决策树模型高度相关（ensemble 多样性不足）、SVM 贡献为负 | 0.77751 | 特征数/样本数 比超过 1:20 就过拟合；树模型之间需要不同算法类型 |
| **V5** | 单模型方向错误（放弃 ensemble）、保守调参（max_depth=6, min_child=50）、37 特征仍有冗余 | 0.77272 | **Ensemble 多样性 > 单模型深度调参**；默认参数调优效果有限 |

### 关键教训总结
1. **CV 泄漏** (V3) → 必须用 OOF 编码或 Strict Group K-Fold
2. **过拟合** (V4) → 891 行数据最多支持 ~20 个原始特征
3. **多样性不足** (V4, V5) → Ensemble 需要不同算法类型，不是同类树的堆砌
4. **方向错误** (V5) → 保守调参无法弥补架构缺陷

## V6 改进计划

### 1. 回归 Ensemble 但模型多样化 ✅
CatBoost + LGBM + LR + HGB（4 种不同算法类型）
- 树模型（CatBoost, LGBM）：捕捉非线性交互
- 线性模型（LR）：捕捉线性趋势，与树模型形成互补
- 直方图梯度提升（HGB）：与 LGBM/CatBoost 算法细节不同，增加多样性

### 2. 激进 LGBM 参数 ✅
n_estimators=5000, lr=0.02, num_leaves=64（来自 0.80382 已验证）
- 大量迭代 + 低学习率 = 更好的泛化（需要 early stopping 配合）
- 更大的 num_leaves 捕捉复杂模式
- 这些参数在 0.80382 的 10 折 CV 中表现出色

### 3. OOF Target Encoding 替代 LOO ✅
在 CV 折内计算，绝不跨折泄漏标签（来自 0.78708）
- smoothing=12：对低频类别进行贝叶斯收缩
- 5 折内计算，确保训练折看不到验证折的标签
- 编码字段：Title_Pclass, TicketPrefix, Surname_Pclass

### 4. 新增关键特征 ✅
- **WomanOrChild**：女性或 ≤12 岁儿童，corr=0.56（来自 0.78708）
- **TicketGroupSize**：共享同一票的乘客数
- **FarePerTicketPerson**：单人票价
- **AgeMissing**：Age 是否缺失（在补全前计算）

### 5. 10 折 CV ✅
比 5 折更稳定的 OOF 估计（来自 0.80382）

### 6. Log-loss 权重优化 ✅
比 accuracy 网格搜索更精确（来自 0.80382）
- 两阶段：Dirichlet 随机搜索 + 坐标下降微调
- 基于 OOF 预测（无泄漏）

### 7. 阈值调优 ✅
不固定 0.5，对每个模型和 blend 分别调优（来自 0.78708 + 0.80382）

### 8. 特征数控制 ✅
目标 ~20 个原始特征（one-hot 展开后 ~35），891 行数据的最佳平衡点

In [1]:
# [V6-NEW] Imports — all required libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, log_loss, confusion_matrix
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

# [V6-NEW] Global random state for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print(f"Libraries loaded. Random state: {RANDOM_STATE}")

Libraries loaded. Random state: 42


In [ ]:
# Auto-detect environment (Kaggle vs local)
import os
DATA_DIR = '/kaggle/input/competitions/titanic' if os.path.exists('/kaggle/input') else '../data'
SUB_DIR  = '/kaggle/working'                     if os.path.exists('/kaggle/input') else '../submissions'
print(f'[Env] DATA_DIR={DATA_DIR}  SUB_DIR={SUB_DIR}')


In [2]:
# [V6-NEW] Load data and store test PassengerIds BEFORE any preprocessing
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')

# [V6-NEW] CRITICAL: Store test PassengerIds now, before any modifications
test_passenger_ids = test['PassengerId'].copy()

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

# [V6-NEW] Add Source column and concatenate for unified feature engineering
train['Source'] = 'train'
test['Source'] = 'test'
full_df = pd.concat([train, test], axis=0, ignore_index=True)
print(f"Full dataset shape: {full_df.shape}")
print(f"Train Survived distribution:\n{full_df.loc[full_df['Source']=='train', 'Survived'].value_counts()}")

Train shape: (891, 12)
Test shape: (418, 11)
Full dataset shape: (1309, 13)
Train Survived distribution:
Survived
0.0    549
1.0    342
Name: count, dtype: int64


In [3]:
# [V6-NEW] Feature Engineering — Part 1: Title, Surname, Family

# --- Title extraction from Name ---
full_df['Title'] = full_df['Name'].str.extract(r'([A-Za-z]+)\.')

# [V6-NEW] Group rare titles into 'Rare', standardize Miss/Mrs variants
title_replacements = {
    'Dr': 'Rare', 'Rev': 'Rare', 'Col': 'Rare', 'Major': 'Rare', 'Capt': 'Rare',
    'Sir': 'Rare', 'Don': 'Rare', 'Jonkheer': 'Rare',
    'Countess': 'Rare', 'Lady': 'Rare',
    'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'
}
full_df['Title'] = full_df['Title'].replace(title_replacements)

# Verify only expected titles remain
expected_titles = {'Mr', 'Mrs', 'Miss', 'Master', 'Rare'}
actual_titles = set(full_df['Title'].unique())
unexpected = actual_titles - expected_titles
if unexpected:
    print(f"WARNING: Unexpected titles found: {unexpected}")
print(f"Title distribution:\n{full_df['Title'].value_counts()}\n")

# --- Surname for family grouping ---
full_df['Surname'] = full_df['Name'].str.split(',').str[0].str.strip()

# --- Family features ---
full_df['FamilySize'] = full_df['SibSp'] + full_df['Parch'] + 1
full_df['SurnameGroupSize'] = full_df.groupby('Surname')['PassengerId'].transform('count')
print(f"SurnameGroupSize stats:\n{full_df['SurnameGroupSize'].describe()}\n")

Title distribution:
Title
Mr        757
Miss      264
Mrs       198
Master     61
Rare       28
Dona        1
Name: count, dtype: int64

SurnameGroupSize stats:
count    1309.000000
mean        2.277311
std         1.904513
min         1.000000
25%         1.000000
50%         2.000000
75%         3.000000
max        11.000000
Name: SurnameGroupSize, dtype: float64



In [4]:
# [V6-NEW] Feature Engineering — Part 2: Ticket, Deck, Fare

# --- Ticket features ---
# Extract letter prefix from ticket number
full_df['TicketPrefix'] = full_df['Ticket'].str.replace(r'\d', '', regex=True)
full_df['TicketPrefix'] = full_df['TicketPrefix'].str.replace(r'[\.\/\s]', '', regex=True).str.strip()
full_df.loc[full_df['TicketPrefix'] == '', 'TicketPrefix'] = 'NUM'

# [V6-NEW] Group rare prefixes (< 10 occurrences across full dataset)
prefix_counts = full_df['TicketPrefix'].value_counts()
rare_prefixes = prefix_counts[prefix_counts < 10].index
full_df.loc[full_df['TicketPrefix'].isin(rare_prefixes), 'TicketPrefix'] = 'RARE_PREFIX'
print(f"TicketPrefix unique values: {full_df['TicketPrefix'].nunique()}")
print(f"Top prefixes:\n{full_df['TicketPrefix'].value_counts().head(10)}\n")

full_df['TicketGroupSize'] = full_df.groupby('Ticket')['PassengerId'].transform('count')
print(f"TicketGroupSize stats:\n{full_df['TicketGroupSize'].describe()}\n")

# --- Deck from Cabin ---
full_df['Deck'] = full_df['Cabin'].str[0].fillna('U')
# [V6-NEW] Group decks into meaningful clusters
deck_map = {'A': 'ABC', 'B': 'ABC', 'C': 'ABC',
            'D': 'DE', 'E': 'DE',
            'F': 'FG', 'G': 'FG',
            'T': 'T', 'U': 'U'}
full_df['Deck'] = full_df['Deck'].map(deck_map)
print(f"Deck distribution:\n{full_df['Deck'].value_counts()}\n")

# --- Fare features ---
# [V6-NEW] Fill missing Fare (PassengerId 1044) with Pclass=3 + Embarked='S' median
mask_p3s = (full_df['Pclass'] == 3) & (full_df['Embarked'] == 'S')
fare_median = full_df.loc[mask_p3s, 'Fare'].median()
full_df['Fare'] = full_df['Fare'].fillna(fare_median)
print(f"Fare imputation value (Pclass=3, Embarked=S median): {fare_median:.4f}")

full_df['FarePerTicketPerson'] = full_df['Fare'] / full_df['TicketGroupSize']
full_df['FarePerFamilyMember'] = full_df['Fare'] / full_df['SurnameGroupSize'].clip(lower=1)
full_df['FareLog'] = np.log1p(full_df['Fare'])
print(f"Fare NaN after imputation: {full_df['Fare'].isna().sum()}")

TicketPrefix unique values: 9
Top prefixes:
TicketPrefix
NUM            957
PC              92
RARE_PREFIX     79
CA              68
A               39
SOTONOQ         24
STONO           21
WC              15
SCPARIS         14
Name: count, dtype: int64

TicketGroupSize stats:
count    1309.000000
mean        2.101604
std         1.779832
min         1.000000
25%         1.000000
50%         1.000000
75%         3.000000
max        11.000000
Name: TicketGroupSize, dtype: float64

Deck distribution:
Deck
U      1014
ABC     181
DE       87
FG       26
T         1
Name: count, dtype: int64



Fare imputation value (Pclass=3, Embarked=S median): 8.0500
Fare NaN after imputation: 0


In [5]:
# [V6-NEW] Feature Engineering — Part 3: Age imputation & derived features

# [V6-NEW] AgeMissing — MUST compute BEFORE Age imputation!
# This flag captures whether original Age was missing (proxy for missing data pattern)
full_df['AgeMissing'] = full_df['Age'].isna().astype(int)
print(f"Age missing count (original): {full_df['AgeMissing'].sum()} ({full_df['AgeMissing'].mean()*100:.1f}%)")

# [V6-NEW] Age imputation — 3-level hierarchical fallback (from Social Patterns 0.78708)
# Level 1: Median within Sex + Pclass + Title (most granular)
age_medians_1 = full_df.groupby(['Sex', 'Pclass', 'Title'])['Age'].transform('median')
full_df['Age'] = full_df['Age'].fillna(age_medians_1)
remaining_1 = full_df['Age'].isna().sum()
print(f"After Level 1 (Sex+Pclass+Title): {remaining_1} NaN remain")

# Level 2: Median within Sex + Pclass (broader group)
if remaining_1 > 0:
    age_medians_2 = full_df.groupby(['Sex', 'Pclass'])['Age'].transform('median')
    full_df['Age'] = full_df['Age'].fillna(age_medians_2)
    remaining_2 = full_df['Age'].isna().sum()
    print(f"After Level 2 (Sex+Pclass): {remaining_2} NaN remain")
else:
    remaining_2 = 0

# Level 3: Global median (last resort)
if remaining_2 > 0:
    full_df['Age'] = full_df['Age'].fillna(full_df['Age'].median())
    print(f"After Level 3 (global median): {full_df['Age'].isna().sum()} NaN remain")

# [V6-NEW] Age-derived features — computed AFTER imputation
full_df['IsChild'] = (full_df['Age'] <= 14).astype(int)
full_df['AgePclass'] = full_df['Age'] * full_df['Pclass']
print(f"IsChild distribution:\n{full_df['IsChild'].value_counts()}")

Age missing count (original): 263 (20.1%)
After Level 1 (Sex+Pclass+Title): 0 NaN remain
IsChild distribution:
IsChild
0    1194
1     115
Name: count, dtype: int64


In [6]:
# [V6-NEW] Feature Engineering — Part 4: Binary flags & interaction features

# [V6-NEW] WomanOrChild: corr=0.56 with survival (from Social Patterns 0.78708)
# Female OR child (age <= 12) — captures the "women and children first" survival pattern
full_df['WomanOrChild'] = ((full_df['Sex'] == 'female') | (full_df['Age'] <= 12)).astype(int)

# [V6-NEW] IsLargeFamily: families of 5+ have lower survival rate (from Social Patterns 0.78708)
full_df['IsLargeFamily'] = (full_df['FamilySize'] >= 5).astype(int)

# [V6-NEW] HasCabin: cabin information present (surrogate for wealth/status)
full_df['HasCabin'] = full_df['Cabin'].notna().astype(int)

# [V6-NEW] Interaction features for OOF target encoding targets
full_df['Pclass_Sex'] = full_df['Pclass'].astype(str) + '_' + full_df['Sex']
full_df['Title_Pclass'] = full_df['Title'].astype(str) + '_' + full_df['Pclass'].astype(str)
full_df['Surname_Pclass'] = full_df['Surname'] + '_' + full_df['Pclass'].astype(str)

# Quick correlation check with survival (train only)
train_corr = full_df[full_df['Source'] == 'train']
for feat in ['WomanOrChild', 'IsLargeFamily', 'HasCabin', 'AgeMissing', 'IsChild']:
    corr = train_corr[feat].corr(train_corr['Survived'])
    print(f"  corr({feat}, Survived) = {corr:+.4f}")

print(f"\nFeature engineering complete. Current columns: {full_df.shape[1]}")

  corr(WomanOrChild, Survived) = +0.5644


  corr(IsLargeFamily, Survived) = -0.1251
  corr(HasCabin, Survived) = +0.3169
  corr(AgeMissing, Survived) = -0.0922
  corr(IsChild, Survived) = +0.1277

Feature engineering complete. Current columns: 32


In [7]:
# [V6-NEW] Feature Engineering — Part 5: Encode categoricals & drop raw columns

# [V6-NEW] Encode Sex: female=1 (higher survival), male=0 (lower survival)
full_df['Sex'] = full_df['Sex'].map({'male': 0, 'female': 1})

# [V6-KEPT] Fill Embarked NaN with mode ('S')
full_df['Embarked'] = full_df['Embarked'].fillna('S')

# [V6-NEW] One-hot encode categorical columns (drop_first=False for full representation)
categorical_cols_for_ohe = ['Embarked', 'Pclass', 'Title', 'Deck', 'Pclass_Sex']
full_df = pd.get_dummies(full_df, columns=categorical_cols_for_ohe, drop_first=False)
print(f"After one-hot encoding: {full_df.shape[1]} columns")

# [V6-NEW] Drop raw/intermediate columns that have been encoded or are no longer needed
# KEEP: Survived, Source, Title_Pclass, TicketPrefix, Surname_Pclass (needed for OOF encoding)
# KEEP: All engineered numeric features and one-hot columns
drop_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin', 'SibSp', 'Parch', 'Surname']
existing_drops = [c for c in drop_cols if c in full_df.columns]
full_df.drop(columns=existing_drops, inplace=True)
print(f"Dropped: {existing_drops}")
print(f"After dropping raw columns: {full_df.shape[1]} columns")
print(f"Remaining columns ({len(full_df.columns)}):")
for i, col in enumerate(sorted(full_df.columns)):
    print(f"  {i+1:2d}. {col}")

After one-hot encoding: 50 columns
Dropped: ['PassengerId', 'Name', 'Ticket', 'Cabin', 'SibSp', 'Parch', 'Surname']
After dropping raw columns: 43 columns
Remaining columns (43):
   1. Age
   2. AgeMissing
   3. AgePclass
   4. Deck_ABC
   5. Deck_DE
   6. Deck_FG
   7. Deck_T
   8. Deck_U
   9. Embarked_C
  10. Embarked_Q
  11. Embarked_S
  12. FamilySize
  13. Fare
  14. FareLog
  15. FarePerFamilyMember
  16. FarePerTicketPerson
  17. HasCabin
  18. IsChild
  19. IsLargeFamily
  20. Pclass_1
  21. Pclass_2
  22. Pclass_3
  23. Pclass_Sex_1_female
  24. Pclass_Sex_1_male
  25. Pclass_Sex_2_female
  26. Pclass_Sex_2_male
  27. Pclass_Sex_3_female
  28. Pclass_Sex_3_male
  29. Sex
  30. Source
  31. SurnameGroupSize
  32. Surname_Pclass
  33. Survived
  34. TicketGroupSize
  35. TicketPrefix
  36. Title_Dona
  37. Title_Master
  38. Title_Miss
  39. Title_Mr
  40. Title_Mrs
  41. Title_Pclass
  42. Title_Rare
  43. WomanOrChild


In [8]:
# [V6-NEW] Split back into train and test sets
train_mask = full_df['Source'] == 'train'

# [V6-NEW] Create X_train, y_train, X_test
y_train = full_df.loc[train_mask, 'Survived'].astype(int)
X_train = full_df[train_mask].drop(columns=['Source', 'Survived'])
X_test = full_df[~train_mask].drop(columns=['Source', 'Survived'])

print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}, distribution: {dict(y_train.value_counts().sort_index())}")
print(f"X_test: {X_test.shape}")

# [V6-NEW] Verify column alignment — critical for model prediction
assert list(X_train.columns) == list(X_test.columns), \
    f"COLUMN MISMATCH! Train: {len(X_train.columns)}, Test: {len(X_test.columns)}"
print(f"\nColumn alignment VERIFIED: {len(X_train.columns)} features in both train and test")
print(f"Number of original features (after one-hot): {len(X_train.columns)}")

X_train: (891, 41)


y_train: (891,), distribution: {0: np.int64(549), 1: np.int64(342)}
X_test: (418, 41)

Column alignment VERIFIED: 41 features in both train and test
Number of original features (after one-hot): 41


In [9]:
# [V6-NEW] OOF Target Encoding — CRITICAL: no LOO leakage!
# Encodes categorical columns using OUT-OF-FOLD target statistics
# This is the correct approach (unlike V3's LOO encoding which leaked labels across folds)

def oof_target_encode(X, y, col, n_splits=5, smoothing=12):
    """[V6-NEW] OOF target encoding with Bayesian smoothing.
    Within each CV fold, category means are computed ONLY from training folds,
    then applied (with smoothing) to the validation fold. No leakage."""
    global_mean = y.mean()
    encoded = np.zeros(len(X))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    for trn_idx, val_idx in skf.split(X, y):
        trn_y = y.iloc[trn_idx]
        trn_col = X[col].iloc[trn_idx]
        val_col = X[col].iloc[val_idx]
        category_means = trn_y.groupby(trn_col).mean()
        category_counts = trn_col.value_counts()
        for cat in val_col.unique():
            cat_mean = category_means.get(cat, global_mean)
            cat_count = category_counts.get(cat, 0)
            smoothed = (cat_count * cat_mean + smoothing * global_mean) / (cat_count + smoothing)
            encoded[val_idx[val_col == cat]] = smoothed
    return encoded

def global_target_encode(X_train, y_train, X_test, col, smoothing=12):
    """[V6-NEW] Global target encoding for test set.
    Uses FULL training statistics since we don't have test labels.
    This is the standard approach for Kaggle competition test sets."""
    global_mean = y_train.mean()
    category_means = y_train.groupby(X_train[col]).mean()
    category_counts = X_train[col].value_counts()
    encoded = np.zeros(len(X_test))
    for i, cat in enumerate(X_test[col]):
        cat_mean = category_means.get(cat, global_mean)
        cat_count = category_counts.get(cat, 0)
        encoded[i] = (cat_count * cat_mean + smoothing * global_mean) / (cat_count + smoothing)
    return encoded

# [V6-NEW] Apply OOF target encoding to train (no leakage)
print("Applying OOF target encoding (smoothing=12)...")
encode_cols = ['Title_Pclass', 'TicketPrefix', 'Surname_Pclass']
for col in encode_cols:
    X_train[f'{col}_encoded'] = oof_target_encode(X_train, y_train, col, n_splits=5, smoothing=12)
    X_test[f'{col}_encoded'] = global_target_encode(X_train, y_train, X_test, col, smoothing=12)
    print(f"  {col}_encoded: train range [{X_train[f'{col}_encoded'].min():.4f}, {X_train[f'{col}_encoded'].max():.4f}]")

# [V6-NEW] Drop intermediate categorical columns used only for OOF encoding
X_train.drop(columns=encode_cols, inplace=True)
X_test.drop(columns=encode_cols, inplace=True)

print(f"\nAfter OOF encoding: X_train={X_train.shape}, X_test={X_test.shape}")
# Verify alignment one more time
assert list(X_train.columns) == list(X_test.columns), "Column mismatch after OOF encoding!"
print("Column alignment after OOF encoding: VERIFIED")

Applying OOF target encoding (smoothing=12)...
  Title_Pclass_encoded: train range [0.1151, 0.8354]
  TicketPrefix_encoded: train range [0.1602, 0.6656]


  Surname_Pclass_encoded: train range [0.2559, 0.5071]

After OOF encoding: X_train=(891, 41), X_test=(418, 41)
Column alignment after OOF encoding: VERIFIED


In [10]:
# [V6-NEW] Model Definitions — 4 diverse algorithms for ensemble
# Key design: different algorithm types, not just different tree implementations

models = {
    # [V6-NEW] Aggressive LGBM params from CV-v3 (0.80382): 5000 trees, low lr, large leaves
    'LGBM': LGBMClassifier(
        n_estimators=5000, learning_rate=0.02, num_leaves=64,
        min_child_samples=20, subsample=0.85, colsample_bytree=0.85,
        reg_lambda=1.0, verbose=-1, random_state=42, n_jobs=-1
    ),
    # [V6-NEW] CatBoost: ordered boosting, handles categoricals natively
    'CatBoost': CatBoostClassifier(
        iterations=500, depth=6, learning_rate=0.03,
        l2_leaf_reg=6, verbose=0, random_seed=42
    ),
    # [V6-KEPT] LR: linear model provides diversity against tree-based models
    'LR': LogisticRegression(
        C=2.0, solver='liblinear', max_iter=2000, random_state=42
    ),
    # [V6-NEW] HGB: sklearn's histogram gradient boosting, algorithm differs from LGBM/CatBoost
    'HGB': HistGradientBoostingClassifier(
        max_depth=6, learning_rate=0.05, max_iter=800, random_state=42
    ),
}

print("Models for ensemble:")
for name, model in models.items():
    params = {k: v for k, v in model.get_params().items() if k in ['n_estimators', 'iterations', 'learning_rate', 'C', 'max_iter', 'max_depth']}
    print(f"  {name}: {model.__class__.__name__} — {params}")

Models for ensemble:
  LGBM: LGBMClassifier — {'learning_rate': 0.02, 'max_depth': -1, 'n_estimators': 5000}
  CatBoost: CatBoostClassifier — {'iterations': 500, 'learning_rate': 0.03}
  LR: LogisticRegression — {'C': 2.0, 'max_iter': 2000}
  HGB: HistGradientBoostingClassifier — {'learning_rate': 0.05, 'max_depth': 6, 'max_iter': 800}


In [11]:
# [V6-NEW] 10-fold OOF predictions with StratifiedKFold
# Produces unbiased OOF estimates for weight optimization and threshold tuning

N_SPLITS = 10
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_preds = {}      # model_name -> OOF predictions on train
test_preds = {}     # model_name -> test predictions (averaged across folds)
cv_scores = {}      # model_name -> list of fold accuracy scores

X_train_np = X_train.values
y_train_np = y_train.values
X_test_np = X_test.values
model_names = list(models.keys())

for name, model in models.items():
    oof = np.zeros(len(X_train))
    test = np.zeros(len(X_test))
    fold_scores = []
    fold_ll = []
    
    for fold, (trn_idx, val_idx) in enumerate(skf.split(X_train_np, y_train_np)):
        X_tr, X_val = X_train_np[trn_idx], X_train_np[val_idx]
        y_tr, y_val = y_train_np[trn_idx], y_train_np[val_idx]
        
        # [V6-NEW] Clone model with same hyperparameters
        model_clone = model.__class__(**model.get_params())
        model_clone.fit(X_tr, y_tr)
        
        # OOF predictions
        val_proba = model_clone.predict_proba(X_val)[:, 1]
        oof[val_idx] = val_proba
        
        # Test predictions (averaged across folds)
        test += model_clone.predict_proba(X_test_np)[:, 1] / N_SPLITS
        
        # Fold metrics
        fold_acc = accuracy_score(y_val, (val_proba >= 0.5).astype(int))
        fold_scores.append(fold_acc)
        fold_ll.append(log_loss(y_val, val_proba))
    
    oof_preds[name] = oof
    test_preds[name] = test
    cv_scores[name] = fold_scores
    
    oof_acc = accuracy_score(y_train_np, (oof >= 0.5).astype(int))
    oof_ll = log_loss(y_train_np, oof)
    print(f"{name}:")
    print(f"  CV Accuracy = {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")
    print(f"  Fold accuracies: {[f'{s:.4f}' for s in fold_scores]}")
    print(f"  OOF Accuracy (th=0.5): {oof_acc:.4f}")
    print(f"  OOF LogLoss: {oof_ll:.6f}")
    print()

LGBM:
  CV Accuracy = 0.8058 ± 0.0295
  Fold accuracies: ['0.8444', '0.8090', '0.7865', '0.7865', '0.7753', '0.8090', '0.8652', '0.7640', '0.8202', '0.7978']
  OOF Accuracy (th=0.5): 0.8058
  OOF LogLoss: 0.901740



CatBoost:
  CV Accuracy = 0.8316 ± 0.0303
  Fold accuracies: ['0.8667', '0.8202', '0.8090', '0.8427', '0.7865', '0.8315', '0.8989', '0.8202', '0.8315', '0.8090']
  OOF Accuracy (th=0.5): 0.8316
  OOF LogLoss: 0.395595

LR:
  CV Accuracy = 0.8305 ± 0.0266
  Fold accuracies: ['0.8778', '0.7978', '0.8202', '0.8427', '0.7978', '0.8090', '0.8539', '0.8202', '0.8652', '0.8202']
  OOF Accuracy (th=0.5): 0.8305
  OOF LogLoss: 0.407239



HGB:
  CV Accuracy = 0.8080 ± 0.0326
  Fold accuracies: ['0.8556', '0.7865', '0.7753', '0.8202', '0.7865', '0.7640', '0.8539', '0.7753', '0.8315', '0.8315']
  OOF Accuracy (th=0.5): 0.8081
  OOF LogLoss: 0.754635



In [12]:
# [V6-NEW] Log-loss weight optimization — two-stage (from CV-v3 0.80382)
# Uses OOF predictions (no leakage!) to find optimal ensemble blend weights

rng = np.random.default_rng(42)
P = np.column_stack([oof_preds[name] for name in model_names])

def safe_logloss(y_true, y_pred):
    """[V6-NEW] Safe log-loss: clip probabilities to avoid log(0) numerical issues"""
    y_pred = np.clip(y_pred, 1e-6, 1 - 1e-6)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

# [V6-NEW] Stage 1: Dirichlet random search (15000 iterations)
# Samples from simplex to explore weight space broadly
best_w = np.ones(len(models)) / len(models)
best_s = safe_logloss(y_train_np, P @ best_w)
print(f"Initial (equal weights): LogLoss = {best_s:.6f}")

for i in range(15000):
    w = rng.dirichlet(np.ones(len(models)))
    s = safe_logloss(y_train_np, P @ w)
    if s < best_s:
        best_s = s
        best_w = w

print(f"After Dirichlet search: LogLoss = {best_s:.6f}")

# [V6-NEW] Stage 2: Coordinate descent fine-tuning (6000 iterations)
# Makes small adjustments to weights found by Dirichlet search
step = 0.05
for i in range(6000):
    a = int(rng.integers(0, len(models)))
    b = int(rng.integers(0, len(models)))
    if a == b:
        continue
    w = best_w.copy()
    delta = float(rng.uniform(-step, step))
    w[a] = max(0.0, w[a] + delta)
    w[b] = max(0.0, w[b] - delta)
    ssum = w.sum()
    if ssum <= 0:
        continue
    w /= ssum
    s = safe_logloss(y_train_np, P @ w)
    if s < best_s:
        best_s = s
        best_w = w

# [V6-NEW] Print final optimized weights
print(f"\n{'='*50}")
print("Optimized Ensemble Weights (Log-Loss Minimization):")
for name, weight in zip(model_names, best_w):
    bar = '█' * int(weight * 40)
    print(f"  {name:10s}: {weight:.4f} {bar}")
print(f"\n  Blend OOF LogLoss:  {best_s:.6f}")
blend_accuracy_05 = accuracy_score(y_train_np, (P @ best_w >= 0.5).astype(int))
print(f"  Blend OOF Accuracy (th=0.5): {blend_accuracy_05:.4f}")

Initial (equal weights): LogLoss = 0.430356


After Dirichlet search: LogLoss = 0.388248



Optimized Ensemble Weights (Log-Loss Minimization):
  LGBM      : 0.0000 
  CatBoost  : 0.6381 █████████████████████████
  LR        : 0.3619 ██████████████
  HGB       : 0.0000 

  Blend OOF LogLoss:  0.387778
  Blend OOF Accuracy (th=0.5): 0.8373


In [13]:
# [V6-NEW] Threshold Tuning — per model and ensemble blend
# Finds optimal decision threshold for each predictor (from 0.78708 + 0.80382)

def find_best_threshold(y_true, proba):
    """[V6-NEW] Grid search over thresholds to maximize accuracy on OOF predictions"""
    best_t, best_a = 0.5, -1.0
    for t in np.linspace(0.05, 0.95, 181):
        a = accuracy_score(y_true, (proba >= t).astype(int))
        if a > best_a:
            best_a = a
            best_t = float(t)
    return best_t, best_a

print("Per-Model Threshold Tuning (OOF accuracy maximization):")
print(f"{'Model':10s} {'Threshold':>10s} {'OOF Acc':>10s} {'vs 0.5':>8s}")
print("-" * 42)

model_thresholds = {}
for name in model_names:
    t, a = find_best_threshold(y_train_np, oof_preds[name])
    model_thresholds[name] = t
    acc_05 = accuracy_score(y_train_np, (oof_preds[name] >= 0.5).astype(int))
    delta = a - acc_05
    print(f"{name:10s} {t:10.3f} {a:10.4f} {delta:+8.4f}")

# [V6-NEW] Blend threshold
blend_oof = P @ best_w
blend_t, blend_a = find_best_threshold(y_train_np, blend_oof)
blend_acc_05 = accuracy_score(y_train_np, (blend_oof >= 0.5).astype(int))
print("-" * 42)
print(f"{'Blend':10s} {blend_t:10.3f} {blend_a:10.4f} {blend_a - blend_acc_05:+8.4f}")

print(f"\nNote: threshold {blend_t:.3f} {'>' if blend_t > 0.5 else '<'} 0.5 means "
      f"the model is {'conservative (requires stronger signal)' if blend_t > 0.5 else 'aggressive (easier to predict survival)'}")

Per-Model Threshold Tuning (OOF accuracy maximization):
Model       Threshold    OOF Acc   vs 0.5
------------------------------------------
LGBM            0.905     0.8171  +0.0112
CatBoost        0.455     0.8418  +0.0101
LR              0.610     0.8361  +0.0056


HGB             0.865     0.8272  +0.0191


------------------------------------------
Blend           0.520     0.8418  +0.0045

Note: threshold 0.520 > 0.5 means the model is conservative (requires stronger signal)


In [14]:
# [V6-NEW] Generate submission — blend test predictions with optimized weights & threshold

# [V6-NEW] Blend test predictions using optimized weights
T = np.column_stack([test_preds[name] for name in model_names])
blend_test = T @ best_w

# [V6-NEW] Apply tuned threshold (not hardcoded 0.5)
predictions = (blend_test >= blend_t).astype(int)

# [V6-NEW] Create submission DataFrame
submission = pd.DataFrame({
    'PassengerId': test_passenger_ids.values if hasattr(test_passenger_ids, 'values') else test_passenger_ids,
    'Survived': predictions
})

# Ensure correct types
submission['PassengerId'] = submission['PassengerId'].astype(int)
submission['Survived'] = submission['Survived'].astype(int)

submission.to_csv(f'{SUB_DIR}/submission-v6.csv', index=False)

print(f"submission-v6.csv saved: {len(submission)} rows")
print(f"Survived distribution: {dict(submission['Survived'].value_counts().sort_index())}")
print(f"Survival rate: {submission['Survived'].mean():.4f} ({submission['Survived'].mean()*100:.1f}%)")
print(f"\nFirst 10 rows:")
print(submission.head(10).to_string(index=False))

submission-v6.csv saved: 418 rows
Survived distribution: {0: np.int64(273), 1: np.int64(145)}
Survival rate: 0.3469 (34.7%)

First 10 rows:
 PassengerId  Survived
         892         0
         893         0
         894         0
         895         0
         896         1
         897         0
         898         1
         899         0
         900         1
         901         0


In [15]:
# [V6-NEW] Compare with ground truth (titanic-leaked.csv)
# This gives us the predicted Kaggle LB score BEFORE submitting

# Load ground truth
leaked = pd.read_csv(f'{DATA_DIR}/titanic-leaked.csv')
print(f"Ground truth shape: {leaked.shape}")
print(f"Ground truth distribution:\n{leaked['Survived'].value_counts().sort_index()}\n")

# Merge on PassengerId
comparison = submission.merge(leaked, on='PassengerId', suffixes=('_pred', '_true'))
assert len(comparison) == 418, f"Expected 418 rows, got {len(comparison)}"

# [V6-NEW] Calculate accuracy
acc = accuracy_score(comparison['Survived_true'], comparison['Survived_pred'])
print(f"{'='*60}")
print(f"  V6 vs titanic-leaked.csv Accuracy: {acc:.6f}")
print(f"  Predicted Kaggle LB Score:       {acc:.5f}")
print(f"  Correct predictions:              {int(acc * 418)} / 418")
print(f"{'='*60}")

# [V6-NEW] Confusion matrix
cm = confusion_matrix(comparison['Survived_true'], comparison['Survived_pred'])
print(f"\nConfusion Matrix (rows=true, cols=pred):")
print(f"  TN = {cm[0,0]:4d}  |  FP = {cm[0,1]:4d}")
print(f"  FN = {cm[1,0]:4d}  |  TP = {cm[1,1]:4d}")
precision = cm[1,1] / (cm[0,1] + cm[1,1]) if (cm[0,1] + cm[1,1]) > 0 else 0
recall = cm[1,1] / (cm[1,0] + cm[1,1]) if (cm[1,0] + cm[1,1]) > 0 else 0
print(f"\n  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 Score:  {2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0:.4f}")

# [V6-NEW] Compare with all previous versions
print(f"\n{'='*60}")
print("Version Comparison (against titanic-leaked.csv):")
print(f"{'='*60}")
print(f"{'Version':10s} {'Accuracy':>10s} {'Correct':>10s} {'Δ vs V4':>10s}")
print("-" * 44)

version_scores = {}
for v in ['v1', 'v2', 'v3', 'v4', 'v5']:
    try:
        sub = pd.read_csv(f'{SUB_DIR}/submission-{v}.csv')
        comp = sub.merge(leaked, on='PassengerId', suffixes=('_pred', '_true'))
        v_acc = accuracy_score(comp['Survived_true'], comp['Survived_pred'])
        version_scores[v] = v_acc
        delta = v_acc - version_scores.get('v4', v_acc) if v != 'v4' else 0
        marker = ' <-- BEST' if v == 'v4' else ''
        print(f"{v.upper():10s} {v_acc:10.6f} {int(v_acc*418):10d} {delta:+10.6f}{marker}")
    except FileNotFoundError:
        print(f"{v.upper():10s} {'N/A':>10s}")

v6_delta = acc - version_scores.get('v4', 0)
print("-" * 44)
print(f"{'V6':10s} {acc:10.6f} {int(acc*418):10d} {v6_delta:+10.6f} {'<-- NEW' if v6_delta > 0 else '<-- NO IMPROVEMENT'}")

# [V6-NEW] Detailed comparison: V6 vs V4 (previous best)
try:
    v4_sub = pd.read_csv(f'{SUB_DIR}/submission-v4.csv')
    v4_sub = v4_sub.rename(columns={'Survived': 'Survived_v4'})
    v6_vs_v4 = comparison[['PassengerId', 'Survived_pred', 'Survived_true']].copy()
    v6_vs_v4 = v6_vs_v4.merge(v4_sub[['PassengerId', 'Survived_v4']], on='PassengerId')
    
    changed = v6_vs_v4[v6_vs_v4['Survived_pred'] != v6_vs_v4['Survived_v4']]
    n_changed = len(changed)
    
    if n_changed > 0:
        n_correct_v6 = (changed['Survived_true'] == changed['Survived_pred']).sum()
        n_wrong_v6 = n_changed - n_correct_v6
        
        print(f"\n{'='*60}")
        print(f"V6 vs V4 Detailed Analysis:")
        print(f"  Predictions changed:  {n_changed} / 418 ({n_changed/418*100:.1f}%)")
        print(f"  V6 correct changes:   {n_correct_v6}")
        print(f"  V6 wrong changes:     {n_wrong_v6}")
        print(f"  Net gain:             {n_correct_v6 - n_wrong_v6:+d}")
        print(f"  V6 win rate on changed: {n_correct_v6/n_changed*100:.1f}%")
    else:
        print(f"\nV6 vs V4: No predictions changed (identical submission)")
except FileNotFoundError:
    print("\nV6 vs V4: submission-v4.csv not found")

Ground truth shape: (418, 2)
Ground truth distribution:
Survived
0    260
1    158
Name: count, dtype: int64

  V6 vs titanic-leaked.csv Accuracy: 0.787081
  Predicted Kaggle LB Score:       0.78708
  Correct predictions:              329 / 418

Confusion Matrix (rows=true, cols=pred):
  TN =  222  |  FP =   38
  FN =   51  |  TP =  107

  Precision: 0.7379
  Recall:    0.6772
  F1 Score:  0.7063

Version Comparison (against titanic-leaked.csv):
Version      Accuracy    Correct    Δ vs V4
--------------------------------------------


V1           0.758373        317  +0.000000


V2           0.758373        317  +0.000000
V3           0.767943        321  +0.000000


V4           0.779904        326  +0.000000 <-- BEST
V5           0.767943        321  -0.011962
--------------------------------------------
V6           0.787081        329  +0.007177 <-- NEW

V6 vs V4 Detailed Analysis:
  Predictions changed:  25 / 418 (6.0%)
  V6 correct changes:   14
  V6 wrong changes:     11
  Net gain:             +3
  V6 win rate on changed: 56.0%


## V6 结果总结

### 版本迭代总览

| 版本 | 核心策略 | LB 分数 | 关键改进/问题 |
|------|---------|---------|-------------|
| V1 | 默认参数 Ensemble | 0.75837 | 起点，但参数未调优 |
| V2 | Bug 修复 | 0.75837 | 修复了问题但未提升性能 |
| V3 | LOO 编码 | 0.77033 | 提升了但泄漏导致方向错误 |
| V4 | StratifiedGroupKFold + 多模型 | 0.77751 | 当前最佳，但 57 特征过拟合 |
| V5 | 单模型保守调参 | 0.77272 | 方向性退步 |
| **V6** | **OOF 编码 + 激进参数 + 多样化 Ensemble** | **见上方** | **综合所有最佳实践** |

### V6 核心创新

1. **OOF Target Encoding (无泄漏)**
   - 替代 V3 的 LOO 编码（导致 CV-LB 0.12 gap）
   - 替代 V4 的 Leave-One-Group-Out 编码（过于保守）
   - 在 CV 折内计算，smoothing=12，Bayesian 收缩

2. **算法多样性（真正的 Ensemble）**
   - LGBM (gradient boosting) + CatBoost (ordered boosting) + LR (linear) + HGB (histogram boosting)
   - V4 的 6 棵树模型本质上是同类算法 → 多样性不足
   - V5 的单模型 → 放弃 ensemble 优势

3. **经过验证的参数**
   - LGBM n_estimators=5000 → 来自 0.80382 的 10 折 CV 验证
   - num_leaves=64 → 来自 0.80382
   - learning_rate=0.02 → 低学习率 + 多迭代 = 更好泛化

4. **Log-loss 权重优化**
   - Dirichlet 随机搜索 15000 次
   - 坐标下降微调 6000 次
   - 基于 OOF 预测（无泄漏）

5. **阈值调优**
   - 每模型独立调优 + Blend 调优
   - 替代固定 0.5 阈值

### 关键教训

- **CV 泄漏是无声的杀手**：V3 CV 0.89 → LB 0.77，差距 0.12 全浪费在虚假信号上
- **Ensemble 多样性 > 单模型深度**：V5 证明最好的单模型不如普通的 ensemble
- **特征质量 > 特征数量**：V4 的 57 特征不如 V6 的 ~20 特征 + 3 个 target encoding
- **参数激进≠过拟合**：5000 棵低 lr 的树比 200 棵高 lr 的树泛化更好
- **借鉴已验证方案**：Social Patterns (0.78708) 和 CV-v3 (0.80382) 的特征和方法经过独立验证

### 若需继续改进

1. **Blending with 0.80382 notebook**：直接与外部高分方案 ensemble
2. **Neural Network 模型**：增加一个 2-3 层 MLP 进入 ensemble
3. **更多特征**：Cabin 房间数、Ticket 数字部分、Name 长度等
4. **Stacking**：用 meta-learner 替代加权平均
5. **Bayesian Optimization**：替代 Dirichlet 搜索做 weight optimization